# OVD-Diagnose — cross-domain OVD failure benchmark

Runs a frozen open-vocabulary detector over a domain in three modes (Global / Oracle /
Agnostic) and computes the (L, S, C) failure fingerprint. See `paper/DESIGN.md`.

**Before running:** Accelerator = GPU, Internet = On.
**Aerial data:** LAE-80C (Locate-Anything-on-Earth benchmark subset), 3592 images, 80 classes.

In [ ]:
# autoreload avoids the stale-module footgun after git pull (no kernel restart needed)
%load_ext autoreload
%autoreload 2
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
# Clone / update repo
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true

In [ ]:
# Deps
!pip install -q ultralytics pycocotools pyyaml

In [ ]:
# Metric self-test (no model / data needed)
!python tools/ovd_diagnose.py

## Download aerial domain (LAE-80C)
Annotations + 1.83 GB image zip from HuggingFace. Skips if already present.

In [ ]:
import os, zipfile, shutil
BASE = '/kaggle/working/YOLOBERT/data/aerial'
ANN_DIR, IMG_DIR = f'{BASE}/annotations', f'{BASE}/images'
os.makedirs(ANN_DIR, exist_ok=True); os.makedirs(IMG_DIR, exist_ok=True)
ann = f'{ANN_DIR}/instances_val.json'
HF = 'https://huggingface.co/datasets/jaychempan/LAE-1M/resolve/main/LAE-80C'

if not os.path.exists(ann):
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark.json?download=true" -O {ann}
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark_categories.json?download=true" -O {ANN_DIR}/categories.json
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark.txt?download=true" -O {ANN_DIR}/classes.txt

if len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')]) < 100:
    zp = '/kaggle/working/images.zip'
    !wget -q --no-check-certificate "{HF}/images.zip?download=true" -O {zp}
    with zipfile.ZipFile(zp) as z:
        z.extractall(IMG_DIR)
    os.remove(zp)
    # flatten a nested images/images/ wrapper if present
    nested = f'{IMG_DIR}/images'
    if os.path.isdir(nested):
        for f in os.listdir(nested):
            shutil.move(f'{nested}/{f}', f'{IMG_DIR}/{f}')
        os.rmdir(nested)

n_imgs = len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')])
print('annotations:', os.path.exists(ann), '| images:', n_imgs)

## Run the diagnostic
One frozen OVD model, three modes, on the aerial domain. `--limit 200` for a quick pass;
drop it for the full 3592-image split.

In [ ]:
!python tools/run_ovd.py \
  --ann  data/aerial/annotations/instances_val.json \
  --imgs data/aerial/images \
  --model yoloworld --weights yolov8s-world.pt \
  --imgz 800 --device cuda:0 \
  --out runs/diag/aerial_yoloworld --limit 200

In [ ]:
# Fingerprint + Paper-1-style IoA-F1 anchor (validates the harness)
import json
from tools.ovd_diagnose import diagnose, ioa_f1, sanity_check
ann = 'data/aerial/annotations/instances_val.json'
d = 'runs/diag/aerial_yoloworld'
G = json.load(open(f'{d}/results_global.json'))
O = json.load(open(f'{d}/results_oracle.json'))

print('== fingerprint (L, S, C) ==')
print(diagnose(ann, G, O))
print('\n== IoA-F1 anchor (target: YOLO-World ~0.03) ==')
print(ioa_f1(ann, G, ioa_thr=0.7, score_thr=0.25))

**Validated result (200-img aerial, YOLO-World):** `S_norm ≈ 0.61` (61% of achievable AP lost to
vocabulary confusion), IoA-F1 ≈ 0.046 ≈ Paper 1's YOLO-World row (0.028). Semantic-confusion +
localization bound, calibration fine.

Next: SAM adapter for a clean `L`, then scale to OWLv2 / Grounding DINO / LLMDet / YOLOE, then agriculture domain.